# Project 1 Phase B — Python Reference for Adaptive Real-Space Median Filtering

This notebook prepares a replaceable 4D-STEM source, saves or verifies the canonical pre-filter benchmark input, and derives and exercises the current public 4Denoise adaptive median behavior without optimizing or changing it.


## Relationship to Phase A and the performance boundary

Project 1 has two phases:

- **Phase A:** fixed `3 × 3` real-space median warm-up.
- **Phase B:** adaptive median with `s=3`, `sMax=7`, the main performance target.

Both begin from the same runtime-derived preparation:

```text
ripple_data_reduced.npy
    → center-disk alignment
    → elliptical correction
    → median_filter_input
```

Scientific preparation is outside median-kernel timing. The source file is replaceable, so numerical dimensions are discovered rather than assumed.


## Paths, imports, versions, and provenance

Run this notebook from its Project 1 directory. The default layout expects the replaceable source one directory above, frozen adaptive fixtures under `reference_data/adaptive_median`, and the canonical generated input under `benchmark_data`.


In [ ]:
from pathlib import Path
import cProfile
import gc
import hashlib
import importlib.util
import inspect
from io import StringIO
import platform
import pstats
import statistics
import time

import numpy as np
import scipy
import skimage


NOTEBOOK_NAME = "python_reference_adaptive_median_filter.ipynb"
AXIS_ORDER = ("scan_y", "scan_x", "detector_y", "detector_x")
PROJECT_DIR = Path.cwd().resolve()
FOURDENOISE_REPO_CANDIDATE = (PROJECT_DIR.parents[1] / "4denoise_git").resolve()
FOURDENOISE_MODULE_CANDIDATE = FOURDENOISE_REPO_CANDIDATE / "fourdenoise.py"

if FOURDENOISE_MODULE_CANDIDATE.exists():
    module_spec = importlib.util.spec_from_file_location(
        "fourdenoise_project1",
        FOURDENOISE_MODULE_CANDIDATE,
    )
    if module_spec is None or module_spec.loader is None:
        raise ImportError(f"Cannot load {FOURDENOISE_MODULE_CANDIDATE}")
    fd = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(fd)
else:
    import fourdenoise as fd

DATA_PATH = (PROJECT_DIR.parent / "ripple_data_reduced.npy").resolve()
REFERENCE_DIR = (PROJECT_DIR / "reference_data" / "adaptive_median").resolve()
BENCHMARK_DIR = (PROJECT_DIR / "benchmark_data").resolve()
BENCHMARK_INPUT_PATH = BENCHMARK_DIR / "median_filter_input.npy"
DATASET_MANIFEST_PATH = BENCHMARK_DIR / "DATASET.md"
FOURDENOISE_MODULE_PATH = Path(fd.__file__).resolve()
alignment_parameters = set(inspect.signature(fd.HyperData.alignment).parameters)
required_alignment_parameters = {
    "method",
    "fit_radius",
    "radius_range",
    "radius_step",
    "iterations",
    "search_radius",
    "enforce_square",
}
if not required_alignment_parameters.issubset(alignment_parameters):
    missing = sorted(required_alignment_parameters - alignment_parameters)
    raise ImportError(
        f"Imported incompatible fourdenoise module {FOURDENOISE_MODULE_PATH}; "
        f"HyperData.alignment is missing parameters {missing}. "
        "Make the current corrected 4Denoise repository importable."
    )
if not hasattr(fd.HyperData, "denoise"):
    raise ImportError(
        f"Imported incompatible fourdenoise module {FOURDENOISE_MODULE_PATH}; "
        "HyperData.denoise is unavailable."
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)
if not REFERENCE_DIR.exists():
    raise FileNotFoundError(REFERENCE_DIR)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

def file_sha256(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_hashes = {
    "reduced_data": file_sha256(DATA_PATH),
    "fourdenoise_module": file_sha256(FOURDENOISE_MODULE_PATH),
}

print(f"Project directory: {PROJECT_DIR}")
print(f"Python {platform.python_version()}")
print(
    f"NumPy {np.__version__}; SciPy {scipy.__version__}; "
    f"scikit-image {skimage.__version__}"
)
print(f"4Denoise module: {FOURDENOISE_MODULE_PATH}")
for name, digest in source_hashes.items():
    print(f"{name} SHA-256: {digest}")


## Load and validate the replaceable reduced data

The explicit upstream axis order is `(scan_y, scan_x, detector_y, detector_x)`; it is provenance, not a guess based on shape. The adaptive benchmark requires at least `sMax` samples along each scan axis so a full maximum neighborhood is meaningful away from boundaries.


In [ ]:
load_started = time.perf_counter()
reduced_source = np.load(DATA_PATH, allow_pickle=False)
load_seconds = time.perf_counter() - load_started

if reduced_source.ndim != 4:
    raise ValueError(
        "Expected a 4D array with axis semantics "
        f"{AXIS_ORDER}; received shape {reduced_source.shape}"
    )
if any(length == 0 for length in reduced_source.shape):
    raise ValueError(f"All four dimensions must be nonempty: {reduced_source.shape}")
if not np.issubdtype(reduced_source.dtype, np.number):
    raise TypeError(f"Expected numeric data, received dtype {reduced_source.dtype}")

source_shape = tuple(int(length) for length in reduced_source.shape)
scan_y, scan_x, detector_y, detector_x = source_shape
if scan_y < 7 or scan_x < 7:
    raise ValueError(
        "Adaptive benchmarking with s=3 and sMax=7 requires scan_y >= 7 "
        f"and scan_x >= 7; received scan shape {(scan_y, scan_x)}"
    )

source_dtype = reduced_source.dtype
source_strides = reduced_source.strides
source_nbytes = reduced_source.nbytes
source_c_contiguous = reduced_source.flags.c_contiguous
source_f_contiguous = reduced_source.flags.f_contiguous
finite_mask = np.isfinite(reduced_source)
input_finite_summary = {
    "nan_count": int(np.isnan(reduced_source).sum()),
    "posinf_count": int(np.isposinf(reduced_source).sum()),
    "neginf_count": int(np.isneginf(reduced_source).sum()),
    "finite_min": float(reduced_source[finite_mask].min()) if finite_mask.any() else np.inf,
    "finite_max": float(reduced_source[finite_mask].max()) if finite_mask.any() else -np.inf,
}
del finite_mask
if any(
    input_finite_summary[name] != 0
    for name in ("nan_count", "posinf_count", "neginf_count")
):
    raise ValueError(f"Adaptive reference requires finite data: {input_finite_summary}")

print(f"current loaded shape: {source_shape}")
print(f"dtype: {source_dtype}")
print(f"axis order (from upstream provenance): {AXIS_ORDER}")
print(f"logical bytes: {source_nbytes:,} ({source_nbytes / 2**20:.3f} MiB)")
print(f"NPY file bytes: {DATA_PATH.stat().st_size:,}")
print(f"strides: {source_strides}")
print(f"C contiguous: {source_c_contiguous}")
print(f"F contiguous: {source_f_contiguous}")
print(f"finite-value summary: {input_finite_summary}")
print(f"load time: {load_seconds:.6f} s")


## Established scientific preparation fitted to the current input

Alignment derives disk centers from the current data and must preserve the scan grid. Detector output extents are data-dependent. Before ellipse correction, the notebook verifies that the complete established `43.5–51.5` pixel annulus fits the aligned detector and that the current mean pattern yields a finite ellipse with axes in that annulus.

An NPY file cannot reveal detector resampling. If a replacement changes detector pixel size, physical origin, or calibrated-ring content, the annulus must be scientifically re-established even if these geometric checks pass.


In [ ]:
alignment_started = time.perf_counter()
aligned_data = fd.HyperData(reduced_source).alignment(
    method="disk",
    fit_radius=True,
    radius_range=[0.1, 3],
    radius_step=0.01,
    iterations=2,
    search_radius=2.5,
    enforce_square=False,
)
alignment_seconds = time.perf_counter() - alignment_started
alignment_metadata = dict(aligned_data.center_beam_metadata)
aligned_shape = tuple(int(length) for length in aligned_data.shape)
if aligned_shape[:2] != source_shape[:2]:
    raise AssertionError(
        f"Alignment changed scan axes: source {source_shape[:2]}, aligned {aligned_shape[:2]}"
    )

del reduced_source
gc.collect()

aligned_detector_shape = aligned_shape[-2:]
aligned_center_y = aligned_detector_shape[0] // 2
aligned_center_x = aligned_detector_shape[1] // 2
complete_annulus_radius = min(
    aligned_center_y,
    aligned_detector_shape[0] - 1 - aligned_center_y,
    aligned_center_x,
    aligned_detector_shape[1] - 1 - aligned_center_x,
)
if complete_annulus_radius < 51.5:
    raise ValueError(
        "The established outer ellipse-fit radius (51.5 detector pixels) "
        f"does not fit in the current aligned detector shape {aligned_detector_shape}. "
        "Re-establish detector calibration and r/R for this replacement source."
    )

mean_dp_before_ellipse = aligned_data.get_dp("mean").array
ellipse_before = aligned_data._extract_ellipse(
    mean_dp_before_ellipse,
    aligned_center_y,
    aligned_center_x,
    43.5,
    51.5,
)
ellipse_before_values = tuple(float(value) for value in ellipse_before)
if not np.isfinite(ellipse_before_values).all():
    raise ValueError(f"Current ellipse fit is nonfinite: {ellipse_before_values}")
if not all(43.5 <= axis <= 51.5 for axis in ellipse_before_values[1:]):
    raise ValueError(
        "Current fitted ellipse axes fall outside the established annulus: "
        f"{ellipse_before_values}. Reassess detector calibration and r/R."
    )

ellipse_started = time.perf_counter()
elliptically_corrected = aligned_data.fix_elliptical_distortions(
    r=43.5,
    R=51.5,
    interp_method="linear",
    return_fix=True,
)
ellipse_seconds = time.perf_counter() - ellipse_started

corrected_shape = tuple(int(length) for length in elliptically_corrected.shape)
mean_dp_after_ellipse = elliptically_corrected.get_dp("mean").array
ellipse_after = elliptically_corrected._extract_ellipse(
    mean_dp_after_ellipse,
    corrected_shape[-2] // 2,
    corrected_shape[-1] // 2,
    43.5,
    51.5,
)
ellipse_after_values = tuple(float(value) for value in ellipse_after)
if corrected_shape != aligned_shape:
    raise AssertionError(
        f"Ellipse correction unexpectedly changed shape: {aligned_shape} -> {corrected_shape}"
    )
if not np.isfinite(ellipse_after_values).all():
    raise ValueError(f"Post-correction ellipse fit is nonfinite: {ellipse_after_values}")

print(f"aligned shape: {aligned_shape}")
print(f"alignment time: {alignment_seconds:.6f} s")
print(f"selected disk radius: {alignment_metadata['radius_px']:.12g} px")
print(f"reference center: {alignment_metadata['reference_center_px']}")
print(f"target center: {alignment_metadata['target_center_px']}")
print(f"final mean center: {alignment_metadata['mean_fit_center_px']}")
print(f"final center std: {alignment_metadata['std_fit_center_px']}")
print(f"ellipse before (angle rad, a px, b px): {ellipse_before_values}")
print(f"ellipse after  (angle rad, a px, b px): {ellipse_after_values}")
print(f"ellipse-corrected shape: {corrected_shape}")
print(f"ellipse-correction time: {ellipse_seconds:.6f} s")


## Canonical adaptive-filter input and axes

`median_filter_input` is the scientifically prepared array immediately before either median operation. Its symbolic layout is `(scan_y_processed, scan_x_processed, detector_y_processed, detector_x_processed)`. For `domain='real'`, each fixed detector coordinate defines an independent scan-space image.


In [ ]:
median_filter_input = np.ascontiguousarray(elliptically_corrected.array)
preprocessing_seconds = alignment_seconds + ellipse_seconds

if median_filter_input.ndim != 4:
    raise AssertionError(f"Prepared array is not 4D: {median_filter_input.shape}")
if median_filter_input.shape[:2] != source_shape[:2]:
    raise AssertionError(
        "Scientific preparation changed scan-grid dimensions: "
        f"{source_shape[:2]} -> {median_filter_input.shape[:2]}"
    )
if not np.issubdtype(median_filter_input.dtype, np.number):
    raise TypeError(f"Prepared dtype is not numeric: {median_filter_input.dtype}")
if not median_filter_input.flags.c_contiguous:
    raise AssertionError("Canonical prepared input must be C-contiguous")

finite_mask = np.isfinite(median_filter_input)
prepared_finite_summary = {
    "nan_count": int(np.isnan(median_filter_input).sum()),
    "posinf_count": int(np.isposinf(median_filter_input).sum()),
    "neginf_count": int(np.isneginf(median_filter_input).sum()),
    "finite_min": float(median_filter_input[finite_mask].min()) if finite_mask.any() else np.inf,
    "finite_max": float(median_filter_input[finite_mask].max()) if finite_mask.any() else -np.inf,
}
del finite_mask
if any(
    prepared_finite_summary[name] != 0
    for name in ("nan_count", "posinf_count", "neginf_count")
):
    raise ValueError(f"Prepared input must be finite: {prepared_finite_summary}")

scan_y_processed, scan_x_processed, detector_y_processed, detector_x_processed = (
    median_filter_input.shape
)
print(f"current prepared shape: {median_filter_input.shape}")
print(f"dtype: {median_filter_input.dtype}")
print(f"axis order: {AXIS_ORDER}")
print(f"strides: {median_filter_input.strides}")
print(f"C contiguous: {median_filter_input.flags.c_contiguous}")
print(f"independent real-space images: {detector_y_processed * detector_x_processed:,}")
print(f"pixel decisions for a full adaptive pass: {median_filter_input.size:,}")
print(f"scientific-preparation time: {preprocessing_seconds:.6f} s")

del mean_dp_before_ellipse, mean_dp_after_ellipse, aligned_data, elliptically_corrected
gc.collect()


## Save and verify the canonical benchmark input

The adaptive notebook independently reproduces the common preparation, saves the canonical input only if bytes changed, reloads it, verifies bitwise identity, and updates `benchmark_data/DATASET.md`. No adaptive-filtered full array is saved.


In [ ]:
def arrays_are_bitwise_equal_by_scan(left, right):
    if left.shape != right.shape or left.dtype != right.dtype:
        return False
    for scan_index in range(left.shape[0]):
        left_bytes = np.asarray(left[scan_index]).view(np.uint8)
        right_bytes = np.asarray(right[scan_index]).view(np.uint8)
        if not np.array_equal(left_bytes, right_bytes):
            return False
    return True

def save_npy_if_changed(path, array):
    if path.exists():
        existing = np.load(path, allow_pickle=False, mmap_mode="r")
        unchanged = arrays_are_bitwise_equal_by_scan(existing, array)
        del existing
        if unchanged:
            return False
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("wb") as stream:
        np.save(stream, array, allow_pickle=False)
    temporary_path.replace(path)
    return True

benchmark_written = save_npy_if_changed(BENCHMARK_INPUT_PATH, median_filter_input)
benchmark_reloaded = np.load(
    BENCHMARK_INPUT_PATH,
    allow_pickle=False,
    mmap_mode="r",
)

if benchmark_reloaded.shape != median_filter_input.shape:
    raise AssertionError("Saved benchmark shape differs from the in-memory prepared array")
if benchmark_reloaded.dtype != median_filter_input.dtype:
    raise AssertionError("Saved benchmark dtype differs from the in-memory prepared array")
if not benchmark_reloaded.flags.c_contiguous:
    raise AssertionError("Saved benchmark input must be C-contiguous")
if not all(
    np.isfinite(np.asarray(benchmark_reloaded[scan_index])).all()
    for scan_index in range(benchmark_reloaded.shape[0])
):
    raise ValueError("Saved benchmark input contains NaN or infinity")
if not arrays_are_bitwise_equal_by_scan(benchmark_reloaded, median_filter_input):
    raise AssertionError("Saved benchmark input is not bit-for-bit identical to memory")

benchmark_hash = file_sha256(BENCHMARK_INPUT_PATH)
benchmark_file_bytes = BENCHMARK_INPUT_PATH.stat().st_size

known_scan_only_source_hash = (
    "3fa102dbc58d1dd4e8cf2caf22cda07b4897edb2dbfec5d921a1c11c42fd30fd"
)
if source_hashes["reduced_data"] == known_scan_only_source_hash:
    detector_provenance = (
        "For this source hash, inspection of the upstream 4Denoise workflow shows "
        "a scan-space crop (`ylim=(0, 85)`, `xlim=(2, 37)`) before saving. "
        "The detector axes were not cropped or resampled, so the established "
        "43.5–51.5 pixel annulus retains its original detector-pixel meaning."
    )
else:
    detector_provenance = (
        "The NPY file alone cannot establish whether detector pixels were cropped, "
        "shifted, or resampled upstream. This run confirmed that the complete "
        "43.5–51.5 pixel annulus fits in the aligned detector grid and that a finite "
        "ellipse was fitted from the current mean diffraction pattern. If the "
        "replacement changed detector sampling or removed the calibrated ring, "
        "re-establish `r` and `R` scientifically before using this artifact."
    )

dataset_manifest = f"""# Project 1 Canonical Benchmark Input

## Purpose

`median_filter_input.npy` is the common scientifically prepared real-data input for future fixed C++, fixed CUDA, adaptive C++, and adaptive CUDA median filtering. It is center-aligned and elliptically corrected, but it is not itself median-filtered.

Small deterministic correctness cases remain under `reference_data/`; this file is the realistic development and performance input.

## Source

- Path: `{DATA_PATH}`
- Current SHA-256: `{source_hashes['reduced_data']}`
- Current runtime shape: `{source_shape}`
- Current dtype: `{source_dtype}`
- Axis interpretation: `{AXIS_ORDER}` (the established upstream 4Denoise convention, not inferred from dimensions)
- C contiguous: `{source_c_contiguous}`
- F contiguous: `{source_f_contiguous}`
- Strides: `{source_strides}` bytes
- Finite: `True`
- Current value range: `[{input_finite_summary['finite_min']}, {input_finite_summary['finite_max']}]`
- Logical bytes: `{source_nbytes}`
- NPY file bytes: `{DATA_PATH.stat().st_size}`

The source file is replaceable and may be overwritten by another reduced version. Every numerical dimension and calibration statement below describes only the artifact generated by this run; it is not a permanent project shape.

## Scientific preprocessing

1. `HyperData.alignment(method='disk', fit_radius=True, radius_range=[0.1, 3], radius_step=0.01, iterations=2, search_radius=2.5, enforce_square=False)`
   - Current fitted disk radius: `{float(alignment_metadata['radius_px'])}` pixels
   - Current reference center: `{tuple(alignment_metadata['reference_center_px'])}`
   - Current target center: `{tuple(alignment_metadata['target_center_px'])}`
   - Current final mean fitted center: `{tuple(alignment_metadata['mean_fit_center_px'])}`
   - Current final fitted-center standard deviation: `{tuple(alignment_metadata['std_fit_center_px'])}`
   - Current aligned shape: `{aligned_shape}`
2. `HyperData.fix_elliptical_distortions(r=43.5, R=51.5, interp_method='linear', return_fix=True)`
   - Current ellipse before correction `(angle_rad, a_px, b_px)`: `{ellipse_before_values}`
   - Current ellipse after correction `(angle_rad, a_px, b_px)`: `{ellipse_after_values}`
   - Current correction output shape: `{median_filter_input.shape}`

{detector_provenance}

Alignment derives disk centers from the current input. Ellipse parameters are fitted from the current aligned mean diffraction pattern. The notebook uses the normal package API; it contains no affine-transform workaround.

## Array contract

The general axis contract is `(scan_y, scan_x, detector_y, detector_x)`. Numerical dimensions are discovered at runtime. Scan axes retain their upstream grid semantics. Alignment can reduce detector extents to a common valid crop; ellipse correction preserves the aligned shape.

Current generated artifact:

- Shape: `{median_filter_input.shape}`
- Dtype: `{median_filter_input.dtype}`
- Axis order: `{AXIS_ORDER}`
- C contiguous: `{median_filter_input.flags.c_contiguous}`
- F contiguous: `{median_filter_input.flags.f_contiguous}`
- Strides: `{median_filter_input.strides}` bytes
- Finite: `True`
- Value range: `[{prepared_finite_summary['finite_min']}, {prepared_finite_summary['finite_max']}]`
- Logical bytes: `{median_filter_input.nbytes}`
- NPY file bytes: `{benchmark_file_bytes}`

## Provenance

- Source SHA-256: `{source_hashes['reduced_data']}`
- Benchmark SHA-256: `{benchmark_hash}`
- Imported 4Denoise module: `{FOURDENOISE_MODULE_PATH}`
- Imported module SHA-256: `{source_hashes['fourdenoise_module']}`
- Scientific procedure: the explicit public 4Denoise calls and runtime-derived values documented above
- Regeneration notebooks: `python_reference_2d_median_filter.ipynb` and `python_reference_adaptive_median_filter.ipynb`
- Notebook that last generated/verified this manifest: `{NOTEBOOK_NAME}`

## Verification

The saved NPY was reloaded as a C-contiguous array and checked for exact shape and dtype, finite values, and bit-for-bit equality with the in-memory `median_filter_input`, one scan slab at a time.

## Regeneration

If `ripple_data_reduced.npy` is overwritten, rerun either Project 1 reference notebook from top to bottom. The notebook refits current alignment and ellipse quantities, regenerates `median_filter_input.npy`, reloads and verifies it, recomputes both SHA-256 values, and updates this file. A replacement that changes detector sampling requires scientific review of the 43.5–51.5 pixel annulus; geometric fit checks alone cannot recover missing physical provenance.

## Performance boundary

Project 1 median benchmarks begin from `median_filter_input.npy`. Source loading, center alignment, ellipse fitting, and elliptical correction are excluded from median-kernel timing. Full filtered reference outputs are intentionally not stored here.
"""
DATASET_MANIFEST_PATH.write_text(dataset_manifest, encoding="utf-8")

print(f"canonical benchmark rewritten: {benchmark_written}")
print(f"canonical path: {BENCHMARK_INPUT_PATH}")
print(f"canonical shape/dtype: {benchmark_reloaded.shape}, {benchmark_reloaded.dtype}")
print(f"canonical strides: {benchmark_reloaded.strides}")
print(f"canonical file bytes: {benchmark_file_bytes:,}")
print(f"canonical SHA-256: {benchmark_hash}")
print("Reloaded canonical array is finite and bit-for-bit identical to memory.")


## Authoritative implementation and public invocation

The public call is:

```python
fd.HyperData(data).denoise(
    method="adaptive_median_filter",
    domain="real",
    s=3,
    sMax=7,
    return_array=True,
)
```

The implementation chain is `HyperData.denoise` → `_DenoiseEngine.apply` → `_DenoiseEngine.denoise` → `_DenoisingMethods.adaptive_median_filter` → `_process_pixel` and, when Stage A succeeds, `_level_b`. Phase B explicitly targets real-space filtering. The module's separate `fix_sign_errors_adaptive` routine is unrelated.


In [ ]:
adaptive_symbols = (
    ("public entry", fd.HyperData.denoise),
    ("4D dispatcher", fd._DenoiseEngine.apply),
    ("method dispatcher", fd._DenoiseEngine.denoise),
    ("adaptive method", fd._DenoisingMethods.adaptive_median_filter),
    ("per-pixel helper", fd._DenoisingMethods._process_pixel),
    ("Stage B helper", fd._DenoisingMethods._level_b),
)

for label, symbol in adaptive_symbols:
    source_file = Path(inspect.getsourcefile(symbol)).resolve()
    source_line = inspect.getsourcelines(symbol)[1]
    print(f"{label}: {symbol.__qualname__} — {source_file}:{source_line}")

method_info = fd.HyperData(np.zeros((3, 3))).denoise_info(
    method="adaptive_median_filter",
    include_doc=False,
    print_info=False,
)
assert method_info["full_signature"] == "adaptive_median_filter(target_data, s=3, sMax=7)"
print(f"method signature: {method_info['full_signature']}")

## Actual algorithm derived from the source

For each 2D image, the method allocates constant padding of width `sMax // 2`. The constant is that image's `np.min` value. It allocates a zero-filled same-shape/same-dtype output and visits every pixel with nested Python loops.

At each pixel, it evaluates centered windows beginning at `s` and increasing the size variable by 2. With the intended odd defaults, the visited windows are `3 × 3`, `5 × 5`, and `7 × 7`.

```text
for each independent 2D image:
    pad by floor(sMax / 2) using that image's global minimum
    allocate same-shape, same-dtype output
    for each y:
        for each x:
            current_size = s
            loop:
                window = centered current_size neighborhood
                Z_min = minimum(window)
                Z_med = median(window)
                Z_max = maximum(window)

                if Z_min < Z_med < Z_max:             # Stage A, strict
                    Z_xy = center(window)
                    if Z_min < Z_xy < Z_max:          # Stage B, strict
                        output[y, x] = Z_xy            # retain center
                    else:
                        output[y, x] = Z_med           # replace center
                    stop this pixel

                current_size += 2
                if current_size > sMax:
                    output[y, x] = center(last_window) # original center
                    stop this pixel
```

The final branch is implementation-specific: when Stage A never succeeds, the method returns the original center—not the last median. Equality with either extreme fails the corresponding strict test. No validation requires positive odd `s`/`sMax`; for an even `s`, the slice expression creates an effective side length of `2 * floor(s/2) + 1`, not `s`. Phase B fixes the supported contract to `s=3`, `sMax=7` rather than inheriting undefined parameter combinations.

With finite `float64` data and odd windows, every result is either the original center or an existing neighborhood median. No interpolation or arithmetic output is introduced, so bit-for-bit validation is appropriate. The implementation uses ordinary `np.min`, `np.median`, and `np.max`, not NaN-aware variants. A NaN makes affected comparisons false; because `np.min(image)` also becomes NaN, padded boundary neighborhoods become affected globally. Infinity follows NumPy comparison/subtraction behavior. Phase B therefore specifies **finite inputs only**.

## Directly observable performance characteristics

- The public real-domain dispatcher loops over every detector coordinate.
- Each 2D adaptive call allocates a padded image and a same-shape output.
- Two more Python loops visit every scan pixel.
- Every attempted window creates a slice view and separately calls `np.min`, `np.median`, and `np.max`.
- `np.median` performs selection/partition machinery for each tiny neighborhood; expanded windows repeat all three statistics from scratch.
- There is no explicit multiprocessing, threading, batching, vectorization, or reuse between neighboring/expanded windows.
- The outer 4D dispatcher also allocates a full-size zero-filled result and copies every 2D result into it.

Which of these costs dominates is a profiling question answered below. Cache behavior, memory-bandwidth limits, allocator cost at scale, and the best future work decomposition remain hypotheses until lower-level profiling is performed.

## Deterministic synthetic branch test

Four independent `7 × 7` real-space planes exercise initial-window retention, initial-window replacement, successful expansion from `3 × 3` to `5 × 5`, and boundary exhaustion through `7 × 7`. This small test is retained because the frozen experimental fixture does not guarantee every control-flow branch. `decision_trace` mirrors predicates only to label the path; the authoritative output comes from the public API.


In [ ]:
def decision_trace(image, y, x, s, sMax):
    padded = np.pad(
        image,
        sMax // 2,
        mode="constant",
        constant_values=np.min(image),
    )
    padded_y = y + sMax // 2
    padded_x = x + sMax // 2
    steps = []
    while True:
        window = padded[
            padded_y - s // 2 : padded_y + s // 2 + 1,
            padded_x - s // 2 : padded_x + s // 2 + 1,
        ]
        minimum = np.min(window)
        median = np.median(window)
        maximum = np.max(window)
        center = window[window.shape[0] // 2, window.shape[1] // 2]
        stage_a = bool(median - minimum > 0 and median - maximum < 0)
        steps.append(
            {
                "s": s,
                "minimum": float(minimum),
                "median": float(median),
                "maximum": float(maximum),
                "center": float(center),
                "stage_a": stage_a,
            }
        )
        if stage_a:
            retain = bool(center - minimum > 0 and center - maximum < 0)
            return center if retain else median, "retain" if retain else "replace", steps
        s += 2
        if s > sMax:
            return center, "exhausted_return_center", steps

synthetic_input = np.empty((7, 7, 1, 4), dtype=np.float64)
synthetic_input[:, :, 0, 0] = np.arange(1, 50, dtype=np.float64).reshape(7, 7)
synthetic_input[:, :, 0, 1] = synthetic_input[:, :, 0, 0]
synthetic_input[3, 3, 0, 1] = 999.0
synthetic_input[:, :, 0, 2] = 10.0
synthetic_input[2:5, 2:5, 0, 2] = 0.0
synthetic_input[3, 3, 0, 2] = 100.0
synthetic_input[:, :, 0, 3] = np.arange(1, 50, dtype=np.float64).reshape(7, 7)
synthetic_input[0, 0, 0, 3] = 999.0
synthetic_before = synthetic_input.copy()

synthetic_output = fd.HyperData(synthetic_input).denoise(
    method="adaptive_median_filter",
    domain="real",
    s=3,
    sMax=7,
    return_array=True,
)

synthetic_cases = {
    "initial retain": (0, 3, 3, 25.0, "retain", [3]),
    "initial replace": (1, 3, 3, 26.0, "replace", [3]),
    "expand then replace": (2, 3, 3, 10.0, "replace", [3, 5]),
    "boundary exhaustion": (3, 0, 0, 999.0, "exhausted_return_center", [3, 5, 7]),
}
for name, (detector_index, sy, sx, expected, expected_action, sizes) in synthetic_cases.items():
    traced_value, action, steps = decision_trace(
        synthetic_input[:, :, 0, detector_index], sy, sx, s=3, sMax=7
    )
    actual = synthetic_output[sy, sx, 0, detector_index]
    assert actual == traced_value == expected
    assert action == expected_action
    assert [step["s"] for step in steps] == sizes
    print(f"{name}: output={actual}, action={action}, windows={sizes}")

assert synthetic_output.shape == synthetic_input.shape
assert synthetic_output.dtype == synthetic_input.dtype
assert np.array_equal(synthetic_input.view(np.uint64), synthetic_before.view(np.uint64))
assert not np.shares_memory(synthetic_output, synthetic_input)
print("All synthetic adaptive-branch assertions passed.")


## Frozen adaptive real-data fixture

The small four-dimensional fixture under `reference_data/adaptive_median/` is preserved as a historical, already-preprocessed correctness contract. Its numerical shape and historical source slices are fixture-specific metadata, not assumptions about the replaceable current source. This notebook reloads and validates it; it does not regenerate it.


In [ ]:
def trace_adaptive_plane(image, s, sMax):
    traced = np.zeros_like(image)
    counts = {
        "pixels": 0,
        "initial_stage_a_success": 0,
        "expanded_stage_a_success": 0,
        "exhausted_return_center": 0,
        "retained_center": 0,
        "replaced_with_median": 0,
        "window_evaluations": 0,
        "boundary_pixels": 0,
        "boundary_pixels_that_expand": 0,
    }
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            value, action, steps = decision_trace(image, y, x, s=s, sMax=sMax)
            traced[y, x] = value
            counts["pixels"] += 1
            counts["window_evaluations"] += len(steps)
            if len(steps) == 1:
                counts["initial_stage_a_success"] += 1
            elif action == "exhausted_return_center":
                counts["exhausted_return_center"] += 1
            else:
                counts["expanded_stage_a_success"] += 1
            if action == "retain":
                counts["retained_center"] += 1
            elif action == "replace":
                counts["replaced_with_median"] += 1
            is_boundary = y in (0, image.shape[0] - 1) or x in (0, image.shape[1] - 1)
            if is_boundary:
                counts["boundary_pixels"] += 1
                if len(steps) > 1:
                    counts["boundary_pixels_that_expand"] += 1
    return traced, counts

reference_input_path = REFERENCE_DIR / "reference_input.npy"
reference_output_path = REFERENCE_DIR / "reference_output_python.npy"
reference_spec_path = REFERENCE_DIR / "REFERENCE_DATA.md"
reference_input = np.load(reference_input_path, allow_pickle=False)
saved_reference_output = np.load(reference_output_path, allow_pickle=False)
reference_input_before = reference_input.copy()

fixture_started = time.perf_counter()
reference_output = fd.HyperData(reference_input).denoise(
    method="adaptive_median_filter",
    domain="real",
    s=3,
    sMax=7,
    return_array=True,
)
fixture_seconds = time.perf_counter() - fixture_started

traced_fixture_output = np.empty_like(reference_input)
fixture_counts = None
for detector_index_y in range(reference_input.shape[2]):
    for detector_index_x in range(reference_input.shape[3]):
        traced_plane, plane_counts = trace_adaptive_plane(
            reference_input[:, :, detector_index_y, detector_index_x],
            s=3,
            sMax=7,
        )
        traced_fixture_output[:, :, detector_index_y, detector_index_x] = traced_plane
        if fixture_counts is None:
            fixture_counts = {name: 0 for name in plane_counts}
        for name, value in plane_counts.items():
            fixture_counts[name] += value

if reference_input.shape != saved_reference_output.shape:
    raise AssertionError("Frozen adaptive fixture input/output shapes differ")
if reference_input.dtype != saved_reference_output.dtype:
    raise AssertionError("Frozen adaptive fixture input/output dtypes differ")
if not np.array_equal(reference_input.view(np.uint64), reference_input_before.view(np.uint64)):
    raise AssertionError("Public adaptive API mutated the frozen fixture input")
if not np.array_equal(reference_output.view(np.uint64), saved_reference_output.view(np.uint64)):
    raise AssertionError("Frozen adaptive fixture no longer matches the public API")
if not np.array_equal(reference_output.view(np.uint64), traced_fixture_output.view(np.uint64)):
    raise AssertionError("Diagnostic trace disagrees with the public adaptive API")

assert fixture_counts["initial_stage_a_success"] > 0
assert fixture_counts["exhausted_return_center"] > 0
assert fixture_counts["retained_center"] > 0
assert fixture_counts["replaced_with_median"] > 0
assert fixture_counts["boundary_pixels_that_expand"] > 0
assert file_sha256(reference_input_path) == "70f9e49082cd2810267925d6539ee2f3201a08c621534b73b552d4bb4079ce93"
assert file_sha256(reference_output_path) == "16a8287748062a713d42e958bdf3a923d29cc31227c1b835e4517ddd73448747"

print(f"frozen fixture shape/dtype: {reference_input.shape}, {reference_input.dtype}")
print(f"fixture time: {fixture_seconds:.6f} s")
print(f"fixture input SHA-256:  {file_sha256(reference_input_path)}")
print(f"fixture output SHA-256: {file_sha256(reference_output_path)}")
print(f"fixture branch counts: {fixture_counts}")


## Timing and profiling on a bounded current-data subset

The comparison uses a centered, C-contiguous subset chosen from the current prepared shape. Each scan dimension is capped at 64 samples and each detector dimension at 8 samples; smaller compatible dimensions are used in full. The exact selected slices and runtime shape are printed below.

Both fixed and adaptive timings use the public 4Denoise API. Neither scientific preparation nor file I/O is included. The full adaptive value is a linear element-count estimate, not a measured full-array result.


In [ ]:
def centered_slice(length, maximum_count):
    count = min(int(length), int(maximum_count))
    start = (int(length) - count) // 2
    return slice(start, start + count)

profile_slices = (
    centered_slice(scan_y_processed, 64),
    centered_slice(scan_x_processed, 64),
    centered_slice(detector_y_processed, 8),
    centered_slice(detector_x_processed, 8),
)
profile_input = np.ascontiguousarray(median_filter_input[profile_slices])
profile_input_before = profile_input.copy()
if profile_input.ndim != 4 or profile_input.shape[0] < 7 or profile_input.shape[1] < 7:
    raise ValueError(f"Profile subset cannot support sMax=7: {profile_input.shape}")

fd.HyperData(profile_input).denoise(
    method="median",
    domain="real",
    window_size=3,
    mode="reflect",
    cval=0.0,
    origin=0,
    return_array=True,
)
fixed_timings = []
for _ in range(9):
    started = time.perf_counter()
    fixed_output = fd.HyperData(profile_input).denoise(
        method="median",
        domain="real",
        window_size=3,
        mode="reflect",
        cval=0.0,
        origin=0,
        return_array=True,
    )
    fixed_timings.append(time.perf_counter() - started)

adaptive_timings = []
adaptive_outputs = []
for _ in range(3):
    started = time.perf_counter()
    adaptive_output = fd.HyperData(profile_input).denoise(
        method="adaptive_median_filter",
        domain="real",
        s=3,
        sMax=7,
        return_array=True,
    )
    adaptive_timings.append(time.perf_counter() - started)
    adaptive_outputs.append(adaptive_output)

assert np.array_equal(profile_input.view(np.uint64), profile_input_before.view(np.uint64))
for output in adaptive_outputs[1:]:
    assert np.array_equal(output.view(np.uint64), adaptive_outputs[0].view(np.uint64))

profiler = cProfile.Profile()
profiler.enable()
profiled_output = fd.HyperData(profile_input).denoise(
    method="adaptive_median_filter",
    domain="real",
    s=3,
    sMax=7,
    return_array=True,
)
profiler.disable()
assert np.array_equal(profiled_output.view(np.uint64), adaptive_outputs[0].view(np.uint64))

profile_stats = pstats.Stats(profiler)
def aggregate_profile_function(function_name):
    values = [
        entry
        for (_, _, name), entry in profile_stats.stats.items()
        if name == function_name
    ]
    return {
        "calls": int(sum(entry[1] for entry in values)),
        "internal_seconds": float(sum(entry[2] for entry in values)),
        "cumulative_seconds": float(sum(entry[3] for entry in values)),
    }

adaptive_seconds = statistics.median(adaptive_timings)
fixed_seconds = statistics.median(fixed_timings)
adaptive_profile = {
    name: aggregate_profile_function(name)
    for name in (
        "adaptive_median_filter",
        "_process_pixel",
        "_level_b",
        "median",
        "min",
        "max",
    )
}
median_fraction = adaptive_profile["median"]["cumulative_seconds"] / profile_stats.total_tt
neighborhood_stat_fraction = (
    adaptive_profile["median"]["cumulative_seconds"]
    + adaptive_profile["min"]["cumulative_seconds"]
    + adaptive_profile["max"]["cumulative_seconds"]
) / profile_stats.total_tt
estimated_full_seconds = adaptive_seconds * median_filter_input.size / profile_input.size

print(f"selected slices: {profile_slices}")
print(f"comparison shape/dtype: {profile_input.shape}, {profile_input.dtype}")
print(f"adaptive timings: {adaptive_timings}")
print(f"adaptive median time: {adaptive_seconds:.6f} s")
print(f"fixed timings: {fixed_timings}")
print(f"fixed median time: {fixed_seconds:.6f} s")
print(f"adaptive/fixed runtime ratio: {adaptive_seconds / fixed_seconds:.3f}×")
print(f"adaptive changed values: {np.count_nonzero(adaptive_outputs[0] != profile_input):,}")
print(f"fixed changed values: {np.count_nonzero(fixed_output != profile_input):,}")
print(f"profiled adaptive time: {profile_stats.total_tt:.6f} s")
print(f"np.median cumulative fraction: {100 * median_fraction:.2f}%")
print(f"median + min + max cumulative fraction: {100 * neighborhood_stat_fraction:.2f}%")
print(f"profile function metrics: {adaptive_profile}")
print(
    "LINEAR ELEMENT-COUNT ESTIMATE ONLY — full current prepared adaptive pass: "
    f"{estimated_full_seconds:.1f} s ({estimated_full_seconds / 60:.1f} min)"
)


In [ ]:
profile_stream = StringIO()
pstats.Stats(profiler, stream=profile_stream).strip_dirs().sort_stats(
    "cumulative"
).print_stats(20)
print(profile_stream.getvalue())

## Interpretation of the measured baseline

Fixed and adaptive outputs have different semantics: fixed median replaces every output with a `3 × 3` median, whereas the adaptive routine may retain a non-extreme center, replace an extreme center, expand the window, or retain the center after exhausting `sMax`.

The current profiling output above identifies costs for the runtime-selected subset. It is evidence about the existing Python reference, not an optimization or a shape-independent speed claim.


# C++ / CUDA Behavioral Contract for Phase B

## Input and axes

- Input is the already prepared `median_filter_input`.
- General shape convention: `(scan_y, scan_x, detector_y, detector_x)`; numerical dimensions are runtime values.
- Filter axes 0 and 1 independently for every fixed detector coordinate.
- Input must be finite and must not be mutated.
- For the initial benchmark contract, `scan_y >= 7` and `scan_x >= 7` are required so a complete `sMax=7` neighborhood is meaningful away from boundaries.

## Parameters and padding

- Initial size variable: `s=3`; maximum: `sMax=7`; growth: `s += 2`.
- Attempt `3 × 3`, `5 × 5`, then `7 × 7` neighborhoods.
- Pad each independent scan-space plane by 3 with that plane's global minimum.

## Decision rules

- Compute `Z_min`, `Z_med`, and `Z_max` for every attempted window.
- Stage A succeeds only when `Z_min < Z_med < Z_max`.
- On success, retain `Z_xy` only when `Z_min < Z_xy < Z_max`; otherwise return `Z_med`.
- If Stage A never succeeds, return the original center from the last attempted window.

The frozen finite-`float64` fixture requires bit-for-bit equality.


## Future CUDA engineering challenge — feasibility, not a design

A natural unit of work is one output element `(scan_y, scan_x, detector_y, detector_x)`, or equivalently one scan pixel within one independent detector-coordinate image. Elements are independent in the output, but each reads overlapping neighborhoods.

Adaptive growth gives threads different amounts of work and can cause warp divergence. The current sample mostly stops at `3 × 3`, so divergence is concentrated at unusual values and boundaries; impulse-heavy data or larger `sMax` could change that sharply. Neighborhood access overlaps heavily, suggesting eventual shared-memory tiling with a halo may help, but C-order 4D layout makes detector coordinates contiguous while real-space neighbors are separated by an entire detector plane. A layout decision or carefully chosen mapping may therefore matter as much as the selection logic.

Repeated min/max/median selection is expensive. Expanded windows currently recompute all statistics rather than reusing the smaller window. Small fixed-size selection networks, incremental statistics, tiling, or other strategies may eventually be evaluated, but only if they preserve the exact contract. Shared-memory tiling would require block-level synchronization; a direct per-thread implementation otherwise needs no cross-output synchronization.

The eventual kernel is likely a mixture of memory-, selection-compute-, and branch-bound behavior. Larger maximum windows increase work and halo traffic roughly with neighborhood area and amplify divergence. The balance must be measured with native CPU and GPU profilers rather than inferred from Python `cProfile`.

## Decision and project naming

Project 1 is organized as `01_4DSTEM_Median_Filter_Acceleration` because the fixed warm-up and adaptive target both remain part of the project. Phase A establishes the controlled implementation workflow; Phase B supplies the main measured performance-engineering problem.


## Remaining TBD items

- Formal benchmark hardware report, warm-up policy, repetition counts, confidence summaries, and problem-size matrix.
- Whether Phase B should support only `float64`, or later add other dtypes with separate contracts.
- Whether non-default odd `(s, sMax)` pairs belong in the first native implementation.
- Native CPU data layout, traversal, parallelization, vectorization/selection strategy, and allocation policy.
- CUDA mapping, data-layout/transposition decision, selection implementation, tiling, divergence measurement, transfers, and crossover sizes.
- Python binding technology and ownership/copy behavior.
- Scientific evaluation across additional adaptive-filter datasets, especially cases with more frequent window growth.

## Final verification

The final cell rechecks source integrity, the canonical benchmark input, frozen adaptive fixtures, the deterministic public-API result, and the presence of the Phase A notebook. It does not create native code or save a full filtered output.


In [ ]:
assert file_sha256(DATA_PATH) == source_hashes["reduced_data"]

final_benchmark = np.load(BENCHMARK_INPUT_PATH, allow_pickle=False, mmap_mode="r")
assert arrays_are_bitwise_equal_by_scan(final_benchmark, median_filter_input)
assert file_sha256(BENCHMARK_INPUT_PATH) == benchmark_hash

final_fixture_input = np.load(reference_input_path, allow_pickle=False)
final_fixture_output = np.load(reference_output_path, allow_pickle=False)
final_recomputed_output = fd.HyperData(final_fixture_input).denoise(
    method="adaptive_median_filter",
    domain="real",
    s=3,
    sMax=7,
    return_array=True,
)
assert np.array_equal(
    final_recomputed_output.view(np.uint64),
    final_fixture_output.view(np.uint64),
)
assert reference_spec_path.exists()
assert DATASET_MANIFEST_PATH.exists()
assert (PROJECT_DIR / "python_reference_2d_median_filter.ipynb").exists()

print("Final adaptive-reference verification passed.")
print(f"Source unchanged: {source_hashes['reduced_data']}")
print(f"Prepared input: {median_filter_input.shape}, {median_filter_input.dtype}")
print(f"Canonical SHA-256: {benchmark_hash}")
print(f"Adaptive fixture input:  {reference_input_path}")
print(f"Adaptive fixture output: {reference_output_path}")
print("Canonical and frozen-fixture checks passed bit for bit.")
print("No C++, CUDA, binding, build, or executable artifact is created here.")

del final_recomputed_output, adaptive_outputs, profiled_output, final_benchmark, benchmark_reloaded
gc.collect()
